# 05 · Does the Format Change the Answer?

An essay published in April 2026 argues that language models fail at questions over
knowledge graphs because a bare list of triples does not look like the text they were
trained on, and that annotating each triple with who asserted it, when and how sure
makes the same graph easier for a model to use. The author says the claim is untested.
This notebook tests one slice of it.

**The prediction under test:** for the same facts and the same questions, accuracy
should rise as the serialisation gets richer.

**What is held fixed:** one synthetic graph (20 wind turbines, their gearboxes,
suppliers and sites), the same questions, temperature 0, and gold answers computed
from the graph rather than written by hand.

**What varies:** the format only.

- `flat`: one short sentence per fact, nothing else
- `annotated`: each fact followed by who asserted it, when, and with what confidence
- `prose`: each fact as a sentence that carries the same who, when and confidence

**What this cannot show.** One synthetic graph, one question set, and only the models
that happen to be running on this machine. A result here is a measurement of those
models on this graph, not a test of the essay's theory.

In [1]:
import random, re, json, math, urllib.request
from collections import defaultdict

rng = random.Random(7)   # the graph, the questions and the gold answers all follow from this seed

SITES = ["Site A", "Site B", "Site C"]
SUPPLIERS = {"Norvane": "Denmark", "Kessler": "Germany", "Aldoro": "Spain", "Hallis": "Denmark"}
TURBINES = [f"T{i}" for i in range(1, 21)]
INSPECTORS = ["Rivera", "Okafor", "Lindqvist", "Tanaka"]

site_of, gearbox_of, supplier_of = {}, {}, {}
for i, t in enumerate(TURBINES):
    site_of[t] = SITES[i % 3]
    gearbox_of[t] = f"G{i + 1}"
    supplier_of[gearbox_of[t]] = rng.choice(list(SUPPLIERS))

facts = []
for t in TURBINES:
    facts.append((t, "is located at", site_of[t]))
    facts.append((t, "has gearbox", gearbox_of[t]))
for g, s in supplier_of.items():
    facts.append((g, "is supplied by", s))
for s, c in SUPPLIERS.items():
    facts.append((s, "is based in", c))

# Who said it, when, and how sure: one record per fact.
prov = {f: (rng.choice(INSPECTORS), f"2026-{rng.randint(1, 9):02d}-{rng.randint(1, 28):02d}",
            rng.choice(["low", "medium", "high"])) for f in facts}

# One deliberate conflict: T5 is also reported at a wrong site, by a different person, with low confidence.
CONFLICT = "T5"
true_site = site_of[CONFLICT]
wrong_site = next(s for s in SITES if s != true_site)
prov[(CONFLICT, "is located at", true_site)] = ("Okafor", "2026-03-02", "high")
wrong_fact = (CONFLICT, "is located at", wrong_site)
facts.append(wrong_fact)
prov[wrong_fact] = ("Tanaka", "2026-06-11", "low")

print(f"{len(TURBINES)} turbines, {len(facts)} facts, 1 planted conflict")

20 turbines, 65 facts, 1 planted conflict


## 1 · Three serialisations of the same facts

The planted conflict matters later: one turbine has two recorded sites, and only the
annotated and prose formats say which record to believe.

In [2]:
def as_flat():
    return "\n".join(f"{s} {p} {o}." for s, p, o in facts)

def as_annotated():
    # the shape RDF-star gives you: a statement, then statements about the statement
    return "\n".join(f"<<{s} | {p} | {o}>> asserted_by {prov[(s, p, o)][0]}; "
                     f"asserted_on {prov[(s, p, o)][1]}; confidence {prov[(s, p, o)][2]}."
                     for s, p, o in facts)

def as_prose():
    return "\n".join(f"{s} {p} {o}, as recorded by {prov[(s, p, o)][0]} on "
                     f"{prov[(s, p, o)][1]} with {prov[(s, p, o)][2]} confidence."
                     for s, p, o in facts)

FORMATS = {"flat": as_flat(), "annotated": as_annotated(), "prose": as_prose()}
for name, text in FORMATS.items():
    print(f"--- {name}: {len(text):,} characters, {len(text.splitlines())} lines")
    print("\n".join(text.splitlines()[:2]))

--- flat: 1,594 characters, 65 lines
T1 is located at Site A.
T1 has gearbox G1.
--- annotated: 6,073 characters, 65 lines
<<T1 | is located at | Site A>> asserted_by Rivera; asserted_on 2026-07-02; confidence low.
<<T1 | has gearbox | G1>> asserted_by Rivera; asserted_on 2026-09-28; confidence low.
--- prose: 5,423 characters, 65 lines
T1 is located at Site A, as recorded by Rivera on 2026-07-02 with low confidence.
T1 has gearbox G1, as recorded by Rivera on 2026-09-28 with low confidence.


## 2 · Two kinds of question

**Relational** questions are answerable from the facts alone, so every format can in
principle answer them. In the annotated and prose formats the extra provenance is
*noise* for these questions, which is exactly what the prediction needs to survive.

**Epistemic** questions ask who recorded something, when, or which of two records to
trust. The flat format has no way to answer them, so it is included only as a control:
a model that answers them from flat records is guessing.

In [3]:
# Gold answers are computed from the graph, so they are correct by construction.
others = [t for t in TURBINES if t != CONFLICT]   # T5's site is deliberately contested
relational = []   # answerable from the facts alone, so answerable in every format
for t in rng.sample(TURBINES, 16):
    relational.append(("two-hop", f"Which country is the supplier of the gearbox in turbine {t} based in?",
                       SUPPLIERS[supplier_of[gearbox_of[t]]]))
for t in rng.sample(TURBINES, 12):
    relational.append(("two-hop", f"Which supplier made the gearbox in turbine {t}?",
                       supplier_of[gearbox_of[t]]))
for s in SUPPLIERS:
    n = sum(1 for t in TURBINES if supplier_of[gearbox_of[t]] == s)
    relational.append(("count", f"How many turbines use a gearbox supplied by {s}?", str(n)))
for t in rng.sample(others, 8):
    relational.append(("lookup", f"Which site is turbine {t} located at?", site_of[t]))

epistemic = []    # answerable only if the provenance is present
for f in rng.sample([f for f in facts if f[1] == "has gearbox"], 6):
    epistemic.append(("who", f"Who recorded that {f[0]} has gearbox {f[2]}?", prov[f][0]))
for f in rng.sample([f for f in facts if f[1] == "is supplied by"], 6):
    epistemic.append(("when", f"On what date was it recorded that {f[0]} is supplied by {f[2]}?", prov[f][1]))
epistemic.append(("conflict", "Turbine T5 has two recorded locations. Which site does the higher-confidence record give?", true_site))
epistemic.append(("conflict", "Turbine T5 has two recorded locations. Who recorded the one with low confidence?", "Tanaka"))

from collections import Counter
print("relational:", len(relational), dict(Counter(k for k, _, _ in relational)))
print("epistemic: ", len(epistemic), dict(Counter(k for k, _, _ in epistemic)))

relational: 40 {'two-hop': 28, 'count': 4, 'lookup': 8}
epistemic:  14 {'who': 6, 'when': 6, 'conflict': 2}


## 3 · Find the models

Any OpenAI-compatible endpoint works: LM Studio, Ollama and llama.cpp's server all
expose one, so no API key is needed. Every chat model found is run.

In [4]:
ENDPOINTS = ["http://localhost:11455/v1", "http://localhost:11434/v1", "http://localhost:1234/v1"]

def find_models():
    # Every chat model on every reachable OpenAI-compatible endpoint. Embedding models are
    # listed too and return HTTP 400 on a chat call, so they are filtered out by name.
    found = []
    for base in ENDPOINTS:
        try:
            with urllib.request.urlopen(f"{base}/models", timeout=3) as r:
                ids = [m["id"] for m in json.loads(r.read()).get("data", [])]
        except Exception:
            continue
        found += [(base, i) for i in ids
                  if not any(w in i.lower() for w in ("embed", "rerank", "whisper", "clip"))]
    return found

MODELS = find_models()
print(f"{len(MODELS)} chat model(s) reachable:", [m for _, m in MODELS] or "none")
if not MODELS:
    print("No local endpoint. Start LM Studio or Ollama and re-run; this notebook has no fallback "
          "because the measurement IS the model's answers.")

1 chat model(s) reachable: ['apple-foundationmodel']


## 4 · Ask every question in every format

In [5]:
SYSTEM = ("Answer using ONLY the records below. Reply with the answer alone: a name, a date, "
          "a country, a site or a number. If the records do not contain the answer, reply exactly: NOT IN RECORDS.")

def ask(base, model, records, question):
    body = json.dumps({"model": model, "temperature": 0, "max_tokens": 40,
                       "messages": [{"role": "system", "content": SYSTEM},
                                    {"role": "user", "content": f"Records:\n{records}\n\nQuestion: {question}"}]}).encode()
    req = urllib.request.Request(f"{base}/chat/completions", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        out = json.loads(r.read())["choices"][0]["message"]["content"]
    return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip()

def correct(answer, gold):
    # whole-token match, so "Site A" is not credited for "Site AB" and "1" is not credited for "12"
    return re.search(r"(?<![\w-])" + re.escape(gold.lower()) + r"(?![\w-])", answer.lower()) is not None

results = {}      # (model, set, format) -> list of (kind, question, gold, answer, ok)
ERRORS = []
for base, model in MODELS:
    for qset, questions, fmts in (("relational", relational, ["flat", "annotated", "prose"]),
                                  ("epistemic", epistemic, ["flat", "annotated", "prose"])):
        for fmt in fmts:
            rows = []
            for kind, q, gold in questions:
                try:
                    a = ask(base, model, FORMATS[fmt], q)
                except Exception as ex:
                    ERRORS.append((model, fmt, q[:40], ex.__class__.__name__))
                    a = "ERROR"
                rows.append((kind, q, gold, a, correct(a, gold)))
            results[(model, qset, fmt)] = rows
print(f"{sum(len(v) for v in results.values())} answers collected, {len(ERRORS)} request errors")

162 answers collected, 0 request errors


## 5 · The result

In [6]:
for model in dict.fromkeys(m for _, m in MODELS):
    print(f"model: {model}")
    print(f"{'':12}" + "".join(f"{f:>14}" for f in ("flat", "annotated", "prose")))
    for qset in ("relational", "epistemic"):
        cells = []
        for fmt in ("flat", "annotated", "prose"):
            rows = results[(model, qset, fmt)]
            cells.append(f"{sum(r[4] for r in rows)}/{len(rows)}")
        print(f"{qset:12}" + "".join(f"{c:>14}" for c in cells))
    print()
    print("relational, by question type (correct/asked)")
    kinds = list(dict.fromkeys(k for k, _, _ in relational))
    for kind in kinds:
        cells = []
        for fmt in ("flat", "annotated", "prose"):
            rows = [r for r in results[(model, "relational", fmt)] if r[0] == kind]
            cells.append(f"{sum(r[4] for r in rows)}/{len(rows)}")
        print(f"{kind:12}" + "".join(f"{c:>14}" for c in cells))
    print()

model: apple-foundationmodel
                      flat     annotated         prose
relational           35/40         29/40         35/40
epistemic             0/14         12/14         12/14

relational, by question type (correct/asked)
two-hop              26/28         19/28         26/28
count                  1/4           2/4           1/4
lookup                 8/8           8/8           8/8



## 6 · Is the gap bigger than the noise?

The same 40 questions went through every format, so the right comparison is paired:
count the questions where exactly one format got it right. A sign test on those
discordant pairs says how often a gap that size would appear if the two formats were
equally good.

In [7]:
def sign_test_p(b, c):
    # exact two-sided sign test on the discordant pairs: b right-only-in-one, c right-only-in-the-other
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)

for model in dict.fromkeys(m for _, m in MODELS):
    print(f"model: {model}   (relational questions, same items in every format)")
    base_rows = results[(model, "relational", "flat")]
    for other in ("annotated", "prose"):
        rows = results[(model, "relational", other)]
        only_flat = sum(1 for a, b in zip(base_rows, rows) if a[4] and not b[4])
        only_other = sum(1 for a, b in zip(base_rows, rows) if b[4] and not a[4])
        print(f"  flat vs {other:9}  right only in flat: {only_flat:2}   right only in {other}: {only_other:2}   "
              f"sign-test p = {sign_test_p(only_flat, only_other):.2f}")

model: apple-foundationmodel   (relational questions, same items in every format)
  flat vs annotated  right only in flat:  8   right only in annotated:  2   sign-test p = 0.11
  flat vs prose      right only in flat:  1   right only in prose:  1   sign-test p = 1.00


## 7 · Look at the misses

An accuracy table cannot say whether a miss was the model misreading the format or
declining to answer, and those lead to different fixes.

In [8]:
for model in dict.fromkeys(m for _, m in MODELS):
    for fmt in ("flat", "annotated", "prose"):
        misses = [r for r in results[(model, "relational", fmt)] if not r[4]]
        print(f"{fmt}: {len(misses)} relational misses" + (" (first four)" if len(misses) > 4 else ""))
        for kind, q, gold, a, _ in misses[:4]:
            print(f"   [{kind}] {q[:62]}  gold={gold!r}  answered={a[:36]!r}")
    print()
    print("epistemic misses on the annotated format:")
    for kind, q, gold, a, ok in results[(model, "epistemic", "annotated")]:
        if not ok:
            print(f"   [{kind}] {q[:62]}  gold={gold!r}  answered={a[:36]!r}")

flat: 5 relational misses (first four)
   [two-hop] Which country is the supplier of the gearbox in turbine T1 bas  gold='Spain'  answered='Denmark'
   [two-hop] Which supplier made the gearbox in turbine T1?  gold='Aldoro'  answered='Norvane'
   [count] How many turbines use a gearbox supplied by Norvane?  gold='10'  answered='8'
   [count] How many turbines use a gearbox supplied by Kessler?  gold='4'  answered='5'
annotated: 11 relational misses (first four)
   [two-hop] Which country is the supplier of the gearbox in turbine T13 ba  gold='Denmark'  answered='Spain'
   [two-hop] Which country is the supplier of the gearbox in turbine T15 ba  gold='Germany'  answered='Denmark'
   [two-hop] Which country is the supplier of the gearbox in turbine T2 bas  gold='Germany'  answered='Denmark'
   [two-hop] Which country is the supplier of the gearbox in turbine T16 ba  gold='Denmark'  answered='Spain'
prose: 5 relational misses (first four)
   [two-hop] Which country is the supplier of the 

## What to take to your own evaluation

- **Hold the content fixed and vary only the format.** The annotated and prose versions
  carry the same facts as the flat one, so any difference is the format and not the
  information.
- **Split the questions by whether the extra information is needed.** On the relational
  questions the provenance is clutter. On the epistemic ones it is the answer. Pooling
  them would hide both results.
- **The relational result did not follow the prediction.** On this model the annotated
  format lost eight questions the flat one got right and won two, the reverse of the
  predicted direction, though a sign test on ten disagreements cannot separate that from
  noise. Prose matched flat.
- **The epistemic gain proves nothing about legibility.** Flat scored zero because the
  answer is not in it. Annotated and prose scored 12 of 14 because the answer is there.
  That is the information being present, not the format being easier to read.
- **This is one model, one synthetic graph and 40 questions.** The counting questions
  were weak in every format. A result like this is a reason to run your own graph through
  your own models with `ENDPOINTS` changed, not a verdict on the essay.